In [ ]:
# 14 · Count 7/10 and above as a like

Take the best **Recall@10** model so far—CF + plots + cast + IMDb ratings—and lower the binary like threshold from **8/10 (4 stars) to 7/10 (3.5 stars)**.

There are two places the threshold matters: the training likes that build the collaborative item features, and the rated movies that form a user's preference profile. The requested model lowers **both**. Two diagnostic variants change one at a time to identify the source of any change.

This is an offline experiment using cached data. No API calls, downloads, edits to earlier notebooks, or app-model changes are needed.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "scripts").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "scripts/like_threshold_experiment.py").is_file(), "Run from this project or its notebooks folder."
sys.path.insert(0, str(ROOT))
(ROOT / "data/.matplotlib").mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / "data/.matplotlib"))

import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image
from threadpoolctl import threadpool_limits
from scripts.like_threshold_experiment import (
    prepare_experiment, run_experiment, THRESHOLDS, BASELINE, REQUESTED,
    COHORT_ALL, COHORT_COMMON,
)
from scripts.export_web_model import file_hash
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 65)


## Prespecified experiment

| Variant | CF training likes | User-profile likes |
|---|---|---|
| Original best model | ≥8/10 | ≥8/10 |
| Profile change only | ≥8/10 | ≥7/10 |
| CF feature change only | ≥7/10 | ≥8/10 |
| **Requested 7+ model** | **≥7/10** | **≥7/10** |

Likes have equal binary weights, including ratings exactly **7/10** in the new definition. No dislike penalty is added.

Keep the original chronological per-user 60/20/20 split, training-candidate catalog and user/item ordering fixed. The CF basis is refit only from training likes; plot TF-IDF, cast features, IMDb quality prior and the selected blend weights stay fixed. Exclude **every** training-rated movie, including ratings below the like threshold. Use the original missing-feature fallbacks, unseen-max scaling, no-like-user scoring, and deterministic ranking ties.

### Measure both definitions of a held-out like

- **8+ relevance:** evaluate every variant against validation ratings ≥4 stars. This reproduces the previous benchmark and gives an apples-to-apples comparison with the original model.
- **7+ relevance:** evaluate every variant against validation ratings ≥3.5 stars. This measures the user's proposed definition. The original model must also be scored using this definition.
- For each definition, use every user with at least one qualifying validation rating. Also report the fixed original 8+ user cohort under both definitions, so any change in the eligible user population is visible.
- Recall and ideal NDCG denominators include **all** qualifying held-out ratings, even movies outside the candidate catalog.

Compare models **within the same evaluation definition and cohort**. A raw score under 7+ relevance is not directly comparable with a score under 8+ relevance: the target, denominators and possibly eligible users change. Even using the same user cohort does not remove the target change.

All four variants are reported. No blend-weight tuning or automatic app promotion occurs. Test feedback is not loaded; this is exploratory validation on data already used in earlier experiments.


In [ ]:
with threadpool_limits(limits=2):
    context = prepare_experiment(ROOT)

overview = []
for threshold in THRESHOLDS:
    train_likes = context["train"].loc[context["train"]["rating"] >= threshold]
    relevant = context["validation"].loc[context["validation"]["rating"] >= threshold]
    overview.append({
        "like threshold / 10": threshold * 2,
        "training likes": len(train_likes),
        "users with training likes": train_likes["userId"].nunique(),
        "validation relevant ratings": len(relevant),
        "validation users with relevant ratings": relevant["userId"].nunique(),
    })
display(pd.DataFrame(overview))
print("Frozen candidate movies:", len(context["ids"]))
print("Frozen selected hybrid weights:", context["metadata"]["weights"])
print("Extra training likes at exactly 7/10:", int(context["train"]["rating"].eq(3.5).sum()))
print("Canonical split:", context["split"]["split_id"])


In [ ]:
result = run_experiment(context=context)
assert result["manifest"]["control_parity"]["verified"]
assert result["manifest"]["test_feedback_read"] is False
assert result["manifest"]["app_model_changed"] is False
assert result["summary"]["variant"].nunique() == 4
for _, group in result["summary"].groupby(["relevance_threshold", "cohort"]):
    assert group["users"].nunique() == 1
    assert group["heldout_positive_events"].nunique() == 1
print("Verified original control for", result["manifest"]["control_parity"]["users"], "validation users.")
print("CF training coverage:")
display(pd.DataFrame(result["manifest"]["cf_coverage"]))
print("Artifacts saved to:", result["output"])


In [ ]:
summary = result["summary"]
all_users = summary.loc[summary["cohort"] == COHORT_ALL].copy()
comparison_rows = []
for relevance in THRESHOLDS:
    frame = all_users.loc[all_users["relevance_threshold"] == relevance].copy()
    original = frame.loc[frame["variant"] == BASELINE].iloc[0]
    frame["recall change vs original"] = frame["mean_recall_at_10"] - original["mean_recall_at_10"]
    frame["ndcg change vs original"] = frame["mean_ndcg_at_10"] - original["mean_ndcg_at_10"]
    display(Markdown(f"### Evaluate against held-out ratings ≥{relevance * 2:g}/10"))
    display(frame[["variant", "users", "heldout_positive_events", "hits_at_10",
                   "mean_recall_at_10", "mean_ndcg_at_10",
                   "recall change vs original", "ndcg change vs original"]].reset_index(drop=True))
    proposed = frame.loc[frame["variant"] == REQUESTED].iloc[0]
    delta = proposed["mean_recall_at_10"] - original["mean_recall_at_10"]
    relative = 100 * delta / original["mean_recall_at_10"]
    comparison_rows.append({
        "held-out like threshold / 10": relevance * 2,
        "users": int(proposed["users"]),
        "original Recall@10": original["mean_recall_at_10"],
        "requested 7+ Recall@10": proposed["mean_recall_at_10"],
        "relative recall change %": relative,
        "original NDCG@10": original["mean_ndcg_at_10"],
        "requested 7+ NDCG@10": proposed["mean_ndcg_at_10"],
    })
    direction = "higher" if delta > 1e-12 else "lower" if delta < -1e-12 else "unchanged"
    display(Markdown(
        f"With **{relevance * 2:g}+ held-out relevance**, the requested 7+ model's Recall@10 is "
        f"**{direction}** than the original model: "
        f"{proposed['mean_recall_at_10']:.6f} versus {original['mean_recall_at_10']:.6f} "
        f"({relative:+.2f}% relative). This compares identical users and relevance labels."
    ))
display(Markdown("### Requested change versus the original model"))
display(pd.DataFrame(comparison_rows))
display(Markdown("### Fixed original user cohort under the 7+ definition"))
display(summary.loc[
    (summary["cohort"] == COHORT_COMMON) & (summary["relevance_threshold"] == 3.5),
    ["variant", "users", "mean_recall_at_10", "mean_ndcg_at_10"]
].reset_index(drop=True))


In [ ]:
paired = result["paired"]
display(paired.loc[
    (paired["variant"] == REQUESTED) & (paired["cohort"] == COHORT_ALL)
].reset_index(drop=True))
display(Markdown(
    "Intervals resample paired users 4,000 times with seed 140. "
    "They are **descriptive**, not a new independent test or a confirmatory significance claim. "
    "All comparisons hold the evaluation relevance definition and user cohort fixed."
))


In [ ]:
labels = {
    "cf_4__profile_4": "Original\nCF 8+, profile 8+",
    "cf_4__profile_3.5": "Profile only\nCF 8+, profile 7+",
    "cf_3.5__profile_4": "CF only\nCF 7+, profile 8+",
    "cf_3.5__profile_3.5": "Requested\nCF 7+, profile 7+",
}
variant_order = list(labels)
colors = ["#4477AA", "#EE9933", "#228833", "#AA3377"]
fig, axes = plt.subplots(2, 2, figsize=(12, 7.5), layout="constrained")
for row, relevance in enumerate(THRESHOLDS):
    values = all_users.loc[all_users["relevance_threshold"] == relevance].set_index("variant").loc[variant_order]
    for col, (metric, name) in enumerate([("mean_recall_at_10", "Recall@10"), ("mean_ndcg_at_10", "NDCG@10")]):
        ax = axes[row, col]
        bars = ax.bar(np.arange(4), values[metric], color=colors)
        ax.set_xticks(np.arange(4), [labels[variant] for variant in variant_order], fontsize=8)
        ax.set(ylabel=name, title=f"Held-out like ≥{relevance * 2:g}/10 · {int(values['users'].iloc[0])} users")
        ax.set_ylim(0, float(values[metric].max()) * 1.18)
        ax.bar_label(bars, fmt="%.4f", padding=3, fontsize=8)
        ax.grid(axis="y", alpha=0.2)
        ax.set_axisbelow(True)
fig.suptitle("Compare models within each panel: evaluation relevance differs between rows", fontsize=12)
chart_path = result["output"] / "threshold_comparison.png"
fig.savefig(chart_path, dpi=150)
plt.close(fig)
display(Image(filename=str(chart_path)))
manifest = result["manifest"]
manifest["files"][chart_path.name] = {"sha256": file_hash(chart_path)}
_ = (result["output"] / "experiment_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8")


## What changes in recommendations?

The examples use the first three user IDs with at least three training ratings exactly **7/10**, chosen without looking at validation gains. Those movies now join the user's liked profile when its threshold is lowered; every previously rated movie remains excluded from recommendations.


In [ ]:
for user in sorted(result["examples"]["userId"].unique()):
    display(Markdown(f"### User {user}: newly included 7/10 likes"))
    display(result["extra_likes"].loc[result["extra_likes"]["userId"] == user,
                                       ["title", "rating_out_of_10"]].reset_index(drop=True))
    picks = result["examples"].loc[result["examples"]["userId"] == user]
    display(picks.pivot(index="rank", columns="variant", values="title"))


## Reading the result

The **both-7+** variant is the requested model. Use the two partial-change variants to see whether any change comes from the collaborative item basis, the user's liked profile, or both.

The prior best model is reproduced at its original 8+ evaluation threshold before accepting any new result. If both-7+ improves scores within a fixed relevance definition, that supports this threshold change for that benchmark. If it only looks better when compared with a historical score using a different definition, that does not establish an improvement.

This dataset's validation feedback has already been reused for several model choices. Current IMDb aggregates can contain later votes than the MovieLens events. Results remain exploratory and retrospective; fresh user feedback would provide a stronger check of live recommendation quality.

The app and earlier notebooks retain their current model and thresholds. Results, full-catalog recommendation IDs for every training user, per-user metrics, paired comparisons, profile coverage, examples and provenance are saved under
`data/processed/per_user_chronological_60_20_20_v1/notebook14_like_threshold/`.

Re-run top-to-bottom in the existing project kernel, or run
`.venv/Scripts/python.exe scripts/like_threshold_experiment.py` from the project root.
